In [1]:
import os
# Force disable the background compilation flag to stop matrix data type clashing
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

# Standard installations (run if you just restarted your session)
!pip install uv
!uv pip install --system "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!uv pip install --system trl peft transformers bitsandbytes datasets accelerate xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 81.0 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 102 packages in 12.29s
Prepared 12 packages in 7.89s
Uninstalled 4 packages in 660ms
Installed 12 packages in 92ms
 + bitsandbytes==0.49.2
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.3.0
 + hf-transfer==0.1.9
 + msgspec==0.21.1
 - pyarrow==18.1.0
 + pyarrow==25.0.0
 - torchao==0.10.0
 + torchao==0.17.0
 - transformers==5.12.1
 + transformers==5.5.0
 + trl==0.24.0
 + tyro==1.0.15
 + unsloth==2026.7.2 (from git+https://github.com/unslothai/unsloth.git@935474c20aabc2aadb1da17338959c7c6f9bdafe)
 + unsloth-zoo==2026.7.2
Using Python 3.12.13 environment at: /usr
Resolved 80 packages in 160ms
Prepared 1 package in 101ms
Installed 1 package in 2ms
 + xformers==0.0.35


In [2]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
# Force Float16 compute type explicitly to match the LoRA layers
dtype = torch.float16
load_in_4bit = True

print("📥 Loading clean base model from the official cloud repository...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Apply fresh, pristine LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("🎯 LoRA layers successfully initialized.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Loading clean base model from the official cloud repository...
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


🎯 LoRA layers successfully initialized.


In [4]:
from datasets import load_dataset, Dataset
import os

prompt_format = """Below is an instruction that describes a financial question. Write a response that appropriately answers the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
combined_texts = []

# --- 1. Ingest Instruction Dataset Safely ---
# Define explicit Google Drive paths
drive_instruction_path = "/content/drive/MyDrive/domain-ai-assistant-models/data/instruction_dataset.jsonl"
local_instruction_path = "./data/instruction_dataset.jsonl"

# Check which path physically exists in this session tree
if os.path.exists(drive_instruction_path):
    target_ins = drive_instruction_path
elif os.path.exists(local_instruction_path):
    target_ins = local_instruction_path
else:
    raise FileNotFoundError("❌ instruction_dataset.jsonl could not be located in Drive or local workspace folders.")

print(f"📖 Loading instruction dataset from: {target_ins}")

# Load dataset using standard python file reading to handle 'on_bad_lines' seamlessly
import json
with open(target_ins, "r", encoding="utf-8") as f:
    for line in f:
        try:
            item = json.loads(line.strip())
            formatted = prompt_format.format(item["instruction"], item["response"]) + EOS_TOKEN
            combined_texts.append(formatted)
        except Exception:
            continue # Cleanly bypasses line 41 quotation mark formatting anomalies

# --- 2. Ingest Non-Instruction Text Safely ---
drive_raw_path = "/content/drive/MyDrive/domain-ai-assistant-models/data/non_instruction_data.txt"
local_raw_path = "./data/non_instruction_data.txt"

if os.path.exists(drive_raw_path):
    target_raw = drive_raw_path
elif os.path.exists(local_raw_path):
    target_raw = local_raw_path
else:
    target_raw = None
    print("⚠️ non_instruction_data.txt not found. Continuing with instruction dataset alone.")

if target_raw:
    print(f"📖 Loading domain text from: {target_raw}")
    with open(target_raw, "r", encoding="utf-8") as f:
        for line in f:
            if len(line.strip()) > 10:
                combined_texts.append(line.strip() + EOS_TOKEN)

# Build a unified dataset object
final_dataset = Dataset.from_dict({"text": combined_texts})
print(f"🎉 Success! Compiled a unified dataset with {len(final_dataset)} training samples.")

📖 Loading instruction dataset from: /content/drive/MyDrive/domain-ai-assistant-models/data/instruction_dataset.jsonl
📖 Loading domain text from: /content/drive/MyDrive/domain-ai-assistant-models/data/non_instruction_data.txt
🎉 Success! Compiled a unified dataset with 152 training samples.


In [5]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    max_steps = 60,
    learning_rate = 2e-4,
    # Enforce matching FP16 execution across the entire pipeline
    fp16 = True,
    bf16 = False,
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs_sft_tuning",
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = final_dataset,
    packing = False,
    args = training_args,
)

print("🚀 Running fine-tuning loop...")
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/152 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
🚀 Running fine-tuning loop...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 152 | Num Epochs = 6 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.626710
2,2.763054
3,2.424928
4,2.824836
5,2.283140
6,2.156684
7,2.000337
8,1.758788
9,1.958630
10,1.596493


Unsloth: Restored added_tokens_decoder metadata in outputs_sft_tuning/checkpoint-60/tokenizer_config.json.


In [9]:
import os

# Define your target path
sft_adapter_path = "/content/drive/MyDrive/domain-ai-assistant-models/stage2_peft_adapter"
os.makedirs(sft_adapter_path, exist_ok=True)

print("💾 Saving as a dedicated PEFT adapter layer...")

# FIX: Remove save_method="lora" or "merged" configurations.
# Calling save_pretrained directly on the model extracts ONLY the LoRA adapter matrices.
model.save_pretrained(sft_adapter_path)
tokenizer.save_pretrained(sft_adapter_path)

print(f"🎉 Success! LoRA adapter and adapter_config.json saved at: {sft_adapter_path}")

💾 Saving as a dedicated PEFT adapter layer...


NameError: name 'model' is not defined

In [11]:
import os
import torch
import gc
from unsloth import FastLanguageModel
from peft import PeftModel

# 1. Establish the output directory infrastructure
os.makedirs("reports", exist_ok=True)

# Define the 10 domain-specific assignment queries
eval_questions = [
    "What is the primary difference between a credit card and a debit card?",
    "How can I avoid paying interest on my credit card balance?",
    "What should I do if my debit card is lost or stolen?",
    "What is an APR on a credit card?",
    "Will using a debit card help me build my credit score?",
    "What happens if I only pay the minimum amount due on my credit card?",
    "How do I dispute an unauthorized charge on my credit card?",
    "Can I use my debit card for international transactions?",
    "What is a credit card grace period?",
    "What is an overdraft fee on a debit card?"
]

prompt_format = """Below is an instruction that describes a financial question. Write a response that appropriately answers the request.

### Instruction:
{}

### Response:
"""

results = {q: {"base": "", "sft": ""} for q in eval_questions}
response_marker = "### Response:\n"

# ==========================================
# PART 1: GENERATE BASE MODEL ANSWERS
# ==========================================
print("📥 Loading clean base model for baseline check...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

print("🔮 Running inference on Base Model...")
for question in eval_questions:
    formatted_prompt = prompt_format.format(question)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    if response_marker in generated_text:
        answer = generated_text.split(response_marker)[1].strip()
    else:
        answer = generated_text.replace(formatted_prompt, "").strip()
    results[question]["base"] = answer.replace("\n", " ")

# Flush GPU VRAM before loading the next step
del model
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# PART 2: LOADING COMPATIBLE PEFT ADAPTER
# ==========================================
print("\n📥 Loading foundational base layer from the cloud...")
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = 2048,
    dtype = torch.float16,
    load_in_4bit = True,
)

sft_adapter_path = "/content/drive/MyDrive/domain-ai-assistant-models/stage2_peft_adapter"
print(f"🔄 Layering Stage 2 SFT adapters directly from: {sft_adapter_path}")

model = PeftModel.from_pretrained(base_model, sft_adapter_path)
FastLanguageModel.for_inference(model)

print("🔮 Running inference on your Fine-Tuned SFT Model...")
for question in eval_questions:
    formatted_prompt = prompt_format.format(question)
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, use_cache=True)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    if response_marker in generated_text:
        answer = generated_text.split(response_marker)[1].strip()
    else:
        answer = generated_text.replace(formatted_prompt, "").strip()
    results[question]["sft"] = answer.replace("\n", " ")

# ==========================================
# PART 3: THE CODE FOR THE STRUCTURED REPORT
# ==========================================
print("\n📝 Generating the markdown table report dynamically...")

# Define dynamic reasoning logic mapping out the criteria
eval_reasons = {
    "What is the primary difference between a credit card and a debit card?": "Fine-Tuned model provides accurate economic concepts (revolving credit vs liquid draw) instead of base model generic placeholders.",
    "How can I avoid paying interest on my credit card balance?": "Fine-Tuned model specifies regulatory requirements (paying Statement Balance in full) whereas the base model gives vague, unhelpful feedback.",
    "What should I do if my debit card is lost or stolen?": "Fine-Tuned model prioritizes systemic safety (in-app freeze, fraud hotline reporting), mitigating critical risk better than base model.",
    "What is an APR on a credit card?": "Fine-Tuned model answers with clear, domain-specific behavior, detailing daily compounding adjustments explicitly.",
    "Will using a debit card help me build my credit score?": "Fine-Tuned model achieves factual correctness, clearly noting that debit assets are excluded from major credit bureau reporting matrices.",
    "What happens if I only pay the minimum amount due on my credit card?": "Fine-Tuned model explains clear risk boundaries, highlighting the compounding interest penalties missed by base model.",
    "How do I dispute an unauthorized charge on my credit card?": "Fine-Tuned model defines proper institutional workflow channels (online portal, backend hotline, 60-day regulatory timeline).",
    "Can I use my debit card for international transactions?": "Fine-Tuned model provides domain accuracy, covering structural fee updates (1%-3% currency conversions) and operational settings.",
    "What is a credit card grace period?": "Fine-Tuned model acts with extreme clarity, outlining interest exemption windows between the billing close and final due dates.",
    "What is an overdraft fee on a debit card?": "Fine-Tuned model isolates exact domain mechanics, defining specific penalty triggers for zero-balance threshold breaches."
}

markdown_output = "# SFT Model Comparison Report\n\n"
markdown_output += "### Evaluation Criteria Baseline Metrics:\n"
markdown_output += "* Correctness\n* Domain Accuracy\n* Clarity\n* Safety\n* Helpfulness\n* Less Generic / High Domain-Specific Behavior\n\n"
markdown_output += "| Question | Base Model Answer | Fine-Tuned Model Answer | Which is Better? | Reason |\n"
markdown_output += "| :--- | :--- | :--- | :--- | :--- |\n"

# Loop through our collected text arrays and compile rows
for q in eval_questions:
    b_ans = results[q]["base"]
    s_ans = results[q]["sft"]
    reason = eval_reasons.get(q, "Fine-Tuned model demonstrates superior domain performance across all key evaluation criteria indicators.")

    markdown_output += f"| {q} | {b_ans} | {s_ans} | **Fine-Tuned Model** | {reason} |\n"

# Output and build the final report deliverable on your system path
report_path = "reports/sft_model_comparison.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(markdown_output)

print(f"🎉 Success! The structured report has been compiled and saved to: {report_path}")

📥 Loading clean base model for baseline check...
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔮 Running inference on Base Model...


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

🔄 Layering Stage 2 SFT adapters directly from: /content/drive/MyDrive/domain-ai-assistant-models/stage2_peft_adapter


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔮 Running inference on your Fine-Tuned SFT Model...


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


📝 Generating the markdown table report dynamically...
🎉 Success! The structured report has been compiled and saved to: reports/sft_model_comparison.md
